# Imports and project paths

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path.cwd().parent
raw_path = project_root / "data" / "raw" / "spy_1m.csv"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Load the raw data

In [2]:
df = pd.read_csv(raw_path)
df.head()

,Datetime,Adj Close,Close,High,Low,Open,Volume
0,NaN,SPY,SPY,SPY,SPY,SPY,SPY
1,2026-03-18 13:30:00+00:00,667.969970703125,667.969970703125,669.0700073242188,667.969970703125,668.3599853515625,4709735
2,2026-03-18 13:31:00+00:00,668.5,668.5,668.5399780273438,667.8800048828125,667.97998046875,345024
3,2026-03-18 13:32:00+00:00,668.8200073242188,668.8200073242188,669.0599975585938,668.25,668.5,265326
4,2026-03-18 13:33:00+00:00,668.6400146484375,668.6400146484375,668.8400268554688,668.47998046875,668.780029296875,204493


# Clean datetime and numeric columns

In [3]:
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
df = df[df["Datetime"].notna()].copy()

numeric_cols = ["Adj Close", "Close", "High", "Low", "Open", "Volume"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.sort_values("Datetime").reset_index(drop=True)
df.dtypes

Datetime     datetime64[us, UTC]
Adj Close                float64
Close                    float64
High                     float64
Low                      float64
Open                     float64
Volume                     int64
dtype: object

b# Set the event and label parameters

In [52]:
k = 5
m = 60
z = 2
cooldown = 10

h0 = 5
hmax = 20
alpha = 2
beta = 2
min_init_move = 0.0002

# Construct returns and event-detection features

In [53]:
df["ret_1"] = df["Close"].pct_change()
df["ret_k"] = df["Close"].pct_change(k)
df["sigma_k"] = df["ret_k"].rolling(m).std()
df["event_zscore"] = df["ret_k"].abs() / df["sigma_k"]

df[["Datetime", "Close", "ret_k", "sigma_k", "event_zscore"]].head(10)

,Datetime,Close,ret_k,sigma_k,event_zscore
0,2026-03-18 13:30:00+00:00,667.969971,NaN,NaN,NaN
1,2026-03-18 13:31:00+00:00,668.500000,NaN,NaN,NaN
2,2026-03-18 13:32:00+00:00,668.820007,NaN,NaN,NaN
3,2026-03-18 13:33:00+00:00,668.640015,NaN,NaN,NaN
4,2026-03-18 13:34:00+00:00,668.770020,NaN,NaN,NaN
5,2026-03-18 13:35:00+00:00,668.849976,0.001317,NaN,NaN
6,2026-03-18 13:36:00+00:00,668.770020,0.000404,NaN,NaN
7,2026-03-18 13:37:00+00:00,669.000000,0.000269,NaN,NaN
8,2026-03-18 13:38:00+00:00,669.229980,0.000882,NaN,NaN
9,2026-03-18 13:39:00+00:00,669.090027,0.000479,NaN,NaN


# Construct additional explanatory features

In [54]:
df["vol_k"] = df["Volume"].rolling(k).sum()
df["vol_k_rel"] = df["vol_k"] / df["vol_k"].rolling(m).mean()
df["range_k"] = df["High"].rolling(k).max() / df["Low"].rolling(k).min() - 1
df["vol_regime"] = df["ret_1"].rolling(m).std()
df["prev_ret_k"] = df["ret_k"].shift(k)

minutes = df["Datetime"].dt.hour * 60 + df["Datetime"].dt.minute
df["mins_from_open"] = minutes - (9 * 60 + 30)

df[["vol_k_rel", "range_k", "vol_regime", "prev_ret_k", "mins_from_open"]].head()

,vol_k_rel,range_k,vol_regime,prev_ret_k,mins_from_open
0,NaN,NaN,NaN,NaN,240
1,NaN,NaN,NaN,NaN,241
2,NaN,NaN,NaN,NaN,242
3,NaN,NaN,NaN,NaN,243
4,NaN,0.001782,NaN,NaN,244


# Detect raw extreme events

In [55]:
raw_event_mask = df["event_zscore"] >= z
raw_event_count = raw_event_mask.sum()

raw_event_count

np.int64(158)

# Apply a cooldown so clustered triggers are not counted repeatedly

In [56]:
selected = np.zeros(len(df), dtype=bool)
last_event_idx = -10**9

for i in range(len(df)):
    if raw_event_mask.iloc[i] and i - last_event_idx > cooldown:
        selected[i] = True
        last_event_idx = i

event_idx = np.flatnonzero(selected)

len(event_idx)

47

# Inspect the detected event times

In [57]:
df.loc[event_idx, ["Datetime", "Close", "ret_k", "event_zscore"]].head(30)

,Datetime,Close,ret_k,event_zscore
184,2026-03-18 16:34:00+00:00,666.349976,-0.001079,2.124295
211,2026-03-18 17:01:00+00:00,665.840027,-0.002008,3.250776
246,2026-03-18 17:36:00+00:00,666.369995,0.001533,2.310121
285,2026-03-18 18:15:00+00:00,665.440002,-0.001201,2.067358
304,2026-03-18 18:34:00+00:00,664.659973,-0.002192,2.847261
322,2026-03-18 18:52:00+00:00,663.848999,-0.001491,2.003784
334,2026-03-18 19:04:00+00:00,664.429993,0.001538,2.061561
390,2026-03-19 13:30:00+00:00,655.849976,-0.009298,7.024645
454,2026-03-19 14:34:00+00:00,658.450012,0.003230,2.379974
501,2026-03-19 15:21:00+00:00,657.590027,-0.003123,2.250685


# Prepare to construct the event dataset

In [58]:
close = df["Close"].to_numpy()
rows = []

# Build the labeled event dataset

In [59]:
for i in event_idx:
    if i + hmax >= len(df):
        continue

    event_move = df.loc[i, "ret_k"]
    event_size = abs(event_move)
    event_sign = int(np.sign(event_move))

    if event_sign == 0:
        continue

    init_post_move = close[i + h0] / close[i] - 1

    if abs(init_post_move) < min_init_move:
        continue

    post_sign = int(np.sign(init_post_move))
    target_move = alpha * event_size

    running_best = 0.0
    max_drawdown_frac = 0.0
    resolved = False
    hit_bar = np.nan
    hit_move = np.nan

    for j in range(h0 + 1, hmax + 1):
        move_from_event = post_sign * (close[i + j] / close[i] - 1)

        if move_from_event > running_best:
            running_best = move_from_event

        if running_best > 0:
            drawdown_frac = (running_best - move_from_event) / running_best
        else:
            drawdown_frac = 0.0

        if drawdown_frac > max_drawdown_frac:
            max_drawdown_frac = drawdown_frac

        if drawdown_frac > beta:
            break

        if move_from_event >= target_move:
            resolved = True
            hit_bar = j
            hit_move = move_from_event
            break

    if not resolved:
        continue

    label_continuation = int(post_sign == event_sign)

    rows.append({
        "Datetime": df.loc[i, "Datetime"],
        "Open": df.loc[i, "Open"],
        "High": df.loc[i, "High"],
        "Low": df.loc[i, "Low"],
        "Close": df.loc[i, "Close"],
        "Volume": df.loc[i, "Volume"],
        "ret_k": df.loc[i, "ret_k"],
        "sigma_k": df.loc[i, "sigma_k"],
        "event_zscore": df.loc[i, "event_zscore"],
        "event_sign": event_sign,
        "event_size": event_size,
        "vol_k_rel": df.loc[i, "vol_k_rel"],
        "range_k": df.loc[i, "range_k"],
        "vol_regime": df.loc[i, "vol_regime"],
        "prev_ret_k": df.loc[i, "prev_ret_k"],
        "mins_from_open": df.loc[i, "mins_from_open"],
        "init_post_move": init_post_move,
        "post_sign": post_sign,
        "target_move": target_move,
        "hit_bar": hit_bar,
        "hit_move": hit_move,
        "max_drawdown_frac": max_drawdown_frac,
        "label_continuation": label_continuation
    })

# Convert the collected rows into the final event dataframe

In [60]:
events = pd.DataFrame(rows)
events = events.dropna().reset_index(drop=True)

events.shape

(7, 23)

# Inspect the final labeled dataset

In [61]:
print(events.shape)
print(events["label_continuation"].value_counts())
events.head()

(7, 23)
label_continuation
1    5
0    2
Name: count, dtype: int64


,Datetime,Open,High,Low,Close,Volume,ret_k,sigma_k,event_zscore,event_sign,...,vol_regime,prev_ret_k,mins_from_open,init_post_move,post_sign,target_move,hit_bar,hit_move,max_drawdown_frac,label_continuation
0,2026-03-19 18:57:00+00:00,657.510010,657.929993,657.419983,657.929993,386101,0.001690,0.000802,2.106223,1,...,0.000360,0.000183,567,0.003937,1,0.003380,7,0.003693,0.000000,1
1,2026-03-20 19:54:00+00:00,647.549988,647.909973,647.239990,647.809998,935642,0.002771,0.001098,2.524240,1,...,0.000432,-0.001391,624,0.001081,1,0.005542,6,0.014966,0.000000,1
2,2026-03-23 15:36:00+00:00,660.289978,660.530029,659.729980,659.739990,329072,-0.002027,0.000923,2.195303,-1,...,0.000434,-0.000219,366,-0.000591,-1,0.004054,20,0.004714,0.310789,1
3,2026-03-23 19:46:00+00:00,658.080017,658.349976,658.080017,658.239990,273565,0.001125,0.000417,2.697973,1,...,0.000245,0.000289,616,-0.001929,-1,0.002251,8,0.002294,0.000000,0
4,2026-03-23 19:57:00+00:00,655.934998,655.979980,655.479980,655.804993,734190,-0.001986,0.000637,3.118574,-1,...,0.000282,-0.001155,627,-0.005467,-1,0.003972,6,0.005131,0.000000,1


# Save the event dataset

In [62]:
events.to_csv(processed_dir / "spy_event_dataset.csv", index=False)

# Basic summary statistics for the current parameter choice

In [63]:
events["label_continuation"].value_counts(normalize=True)
events[["event_size", "init_post_move", "target_move", "hit_bar", "max_drawdown_frac"]].describe()

,event_size,init_post_move,target_move,hit_bar,max_drawdown_frac
count,7.000000,7.000000,7.000000,7.000000,7.000000
mean,0.001934,-0.000246,0.003868,9.428571,0.054450
std,0.000502,0.003154,0.001005,4.961759,0.116036
min,0.001125,-0.005467,0.002251,6.000000,0.000000
25%,0.001725,-0.001695,0.003451,6.500000,0.000000
50%,0.001986,-0.000591,0.003972,8.000000,0.000000
75%,0.002103,0.001895,0.004205,9.500000,0.035180
max,0.002771,0.003937,0.005542,20.000000,0.310789


# Subset of total recognised eventsm

In [107]:
k = 10
m = 120
z = 2
cooldown = 20

h0 = 6
hmax = 400
alpha = 3
beta = 2
min_init_move = 0.00015

In [108]:
terminal_counts = {
    "hit_before_h0": 0,
    "no_established_bias": 0,
    "resolved": 0,
    "not_resolved_within_hmax": 0,
    "hit_max_drawdown": 0,
    "dropped_near_end": 0,
}

for i in event_idx:
    if i + hmax >= len(df):
        terminal_counts["dropped_near_end"] += 1
        continue

    event_price = df.loc[i, "Close"]
    event_size = abs(df.loc[i, "ret_k"])
    target_move = alpha * event_size

    hit_early = False
    for j in range(1, h0 + 1):
        move_abs = abs(df.loc[i + j, "Close"] / event_price - 1)
        if move_abs >= target_move:
            terminal_counts["hit_before_h0"] += 1
            hit_early = True
            break

    if hit_early:
        continue

    init_post_move = df.loc[i + h0, "Close"] / event_price - 1

    if abs(init_post_move) < min_init_move:
        terminal_counts["no_established_bias"] += 1
        continue

    post_sign = np.sign(init_post_move)
    running_best = 0.0
    resolved = False
    stopped_by_drawdown = False

    for j in range(h0 + 1, hmax + 1):
        move_from_event = post_sign * (df.loc[i + j, "Close"] / event_price - 1)

        if move_from_event > running_best:
            running_best = move_from_event

        if running_best > 0:
            drawdown_frac = (running_best - move_from_event) / running_best
            if drawdown_frac > beta:
                terminal_counts["hit_max_drawdown"] += 1
                stopped_by_drawdown = True
                break

        if move_from_event >= target_move:
            terminal_counts["resolved"] += 1
            resolved = True
            break

    if (not resolved) and (not stopped_by_drawdown):
        terminal_counts["not_resolved_within_hmax"] += 1

print("Total number of recognised extreme events:", len(event_idx))
print("Total hit before h0:", terminal_counts["hit_before_h0"])
print("Total with no established bias:", terminal_counts["no_established_bias"])
print("Total resolved:", terminal_counts["resolved"])
print("Total not resolved within hmax:", terminal_counts["not_resolved_within_hmax"])
print("Total hit max drawdown:", terminal_counts["hit_max_drawdown"])
print("Dropped near end of sample:", terminal_counts["dropped_near_end"])

print(
    "Check total:",
    terminal_counts["hit_before_h0"]
    + terminal_counts["no_established_bias"]
    + terminal_counts["resolved"]
    + terminal_counts["not_resolved_within_hmax"]
    + terminal_counts["hit_max_drawdown"]
    + terminal_counts["dropped_near_end"]
)

Total number of recognised extreme events: 47
Total hit before h0: 2
Total with no established bias: 5
Total resolved: 13
Total not resolved within hmax: 1
Total hit max drawdown: 20
Dropped near end of sample: 6
Check total: 47
